In [3]:
print("This is where we are going to master Langchain")

This is where we are going to master Langchain


In [5]:
!pip install langchain langchain-openai langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf

In [6]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')[0:15]

'sk-proj-cmZoZgA'

In [7]:
import os
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')


In [8]:
os.environ["OPENAI_API_KEY"]  = OPENAI_API_KEY

In [9]:
!pip install langchain-anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 37.0 MB/s eta 0:00:00


In [11]:
from langchain.agents import create_agent

In [15]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'It’s currently sunny in San Francisco.'}]


# Models

In [16]:
from langchain.chat_models import init_chat_model

In [17]:
openai_model = init_chat_model("openai:gpt-5-mini")

In [18]:
response = openai_model.invoke("In one line, tell me what is Langchain?")

In [19]:
print(response.content)

LangChain is an open-source framework for building LLM-powered applications, offering abstractions like prompts, chains, agents, memory, and data integrations.


In [20]:
chunks = []
full_message = None
for chunk in openai_model.stream("Write one short sentence about the ocean."):
    chunks.append(chunk)
    print(repr(chunk.text), " <- chunk type:", type(chunk).__name__)
    full_message = chunk if full_message is None else full_message + chunk

print()
print("Reconstructed full message:", full_message.content)


''  <- chunk type: AIMessageChunk
'The'  <- chunk type: AIMessageChunk
' ocean'  <- chunk type: AIMessageChunk
' is'  <- chunk type: AIMessageChunk
' vast'  <- chunk type: AIMessageChunk
' and'  <- chunk type: AIMessageChunk
' full'  <- chunk type: AIMessageChunk
' of'  <- chunk type: AIMessageChunk
' life'  <- chunk type: AIMessageChunk
'.'  <- chunk type: AIMessageChunk
''  <- chunk type: AIMessageChunk
''  <- chunk type: AIMessageChunk
''  <- chunk type: AIMessageChunk

Reconstructed full message: The ocean is vast and full of life.


In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage

messages = [
    SystemMessage(content="You are a pirate, answer everything in pirate language"),
    HumanMessage(content="What is the capital of France?")
]

response = openai_model.invoke(messages)

print(response.content)


Arrr, the capital o' France be Paris, matey!


In [ ]:
import os
from anthropic import Anthropic
from google import genai
from openai import OpenAI

# Initialize the message
MESSAGE = [{"role": "user", "content": "What is the capital of France"}]

# ==========================================
# 1. ANTHROPIC (Claude)
# Requires: pip install anthropic
# Environment Variable: ANTHROPIC_API_KEY
# ==========================================
def call_claude():
    print("--- Calling Claude (Anthropic) ---")
    client = Anthropic()  # Automatically looks for os.environ.get("ANTHROPIC_API_KEY")

    response = client.messages.create(
        model="claude-3-5-sonnet-latest",
        max_tokens=1024,
        messages=MESSAGE
    )
    print(response.content[0].text)


# ==========================================
# 2. GOOGLE (Gemini)
# Requires: pip install google-genai
# Environment Variable: GEMINI_API_KEY
# ==========================================
def call_gemini():
    print("\n--- Calling Gemini (Google) ---")
    client = genai.Client()  # Automatically looks for os.environ.get("GEMINI_API_KEY")

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=MESSAGE[0]["content"],
    )
    print(response.text)


# ==========================================
# 3. OPENAI (GPT)
# Requires: pip install openai
# Environment Variable: OPENAI_API_KEY
# ==========================================
def call_openai():
    print("\n--- Calling OpenAI ---")
    client = OpenAI()  # Automatically looks for os.environ.get("OPENAI_API_KEY")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=MESSAGE
    )
    print(response.choices[0].message.content)


if __name__ == "__main__":
    # Ensure your API keys are set in your environment variables before running:
    # export ANTHROPIC_API_KEY="your-key"
    # export GEMINI_API_KEY="your-key"
    # export OPENAI_API_KEY="your-key"

    try:
        call_claude()
    except Exception as e:
        print(f"Claude Error: {e}")

    try:
        call_gemini()
    except Exception as e:
        print(f"Gemini Error: {e}")

    try:
        call_openai()
    except Exception as e:
        print(f"OpenAI Error: {e}")

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage

messages = [
    SystemMessage(content="You are a pirate, answer everything in pirate language"),
    HumanMessage(content="What is the capital of France?")
]

response = openai_model.invoke(messages)

print(response.content)

Arrr, the capital o' France be Paris, matey!


In [ ]:
print("text :                    ",response.text)
print("content_blocks            ", response.content_blocks)
print("id:                       ", response.id)
print("tool_calls                ", response.tool_calls)

text :                     Arrr, the capital o' France be Paris, matey!
content_blocks             [{'type': 'text', 'text': "Arrr, the capital o' France be Paris, matey!"}]
id:                        lc_run--019f7395-602d-7e71-8ed7-89ec75573389-0
tool_calls                 []


In [ ]:
from pprint import pprint

pprint(response)

AIMessage(content="Arrr, the capital o' France be Paris, matey!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 27, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E2rOIiCgHNAWVNbndAQVHE7e7b5fO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f7395-602d-7e71-8ed7-89ec75573389-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 151, 'total_tokens': 178, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})


In [ ]:
{
    "content": "Arrr, the capital o' France be Paris, matey!",
    "additional_kwargs": {
        "refusal": null
    },
    "response_metadata": {
        "token_usage": {
            "completion_tokens": 151,
            "prompt_tokens": 27,
            "total_tokens": 178,
            "completion_tokens_details": {
                "accepted_prediction_tokens": 0,
                "audio_tokens": 0,
                "reasoning_tokens": 128,
                "rejected_prediction_tokens": 0
            },
            "prompt_tokens_details": {
                "audio_tokens": 0,
                "cache_write_tokens": null,
                "cached_tokens": 0
            }
        },
        "model_provider": "openai",
        "model_name": "gpt-5-mini-2025-08-07",
        "system_fingerprint": null,
        "id": "chatcmpl-E2rOIiCgHNAWVNbndAQVHE7e7b5fO",
        "service_tier": "default",
        "finish_reason": "stop",
        "logprobs": null
    },
    "id": "lc_run--019f7395-602d-7e71-8ed7-89ec75573389-0",
    "tool_calls": [],
    "invalid_tool_calls": [],
    "usage_metadata": {
        "input_tokens": 27,
        "output_tokens": 151,
        "total_tokens": 178,
        "input_token_details": {
            "audio": 0,
            "cache_read": 0
        },
        "output_token_details": {
            "audio": 0,
            "reasoning": 128
        }
    }
}

In [ ]:
response.usage_metadata

{'input_tokens': 27,
 'output_tokens': 151,
 'total_tokens': 178,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 128}}

In [15]:
!pip install -U "langchain-openrouter"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.1/837.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.4 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4


In [16]:
print("Hello")

Hello


In [18]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENROUTER_API_KEY"] = "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"

model = init_chat_model(
    "auto",
    model_provider="openrouter",
)

response = model.invoke("Hello Which model are you ?")

In [ ]:
response

AIMessage(content='I am a large language model, trained by Google.', additional_kwargs={}, response_metadata={'model_name': 'google/gemini-2.5-flash-lite', 'id': 'gen-1784351768-JJB6L5Qt3jsxRCauTXfG', 'created': 1784351768, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 5e-06, 'cost_details': {'upstream_inference_completions_cost': 4.4e-06, 'upstream_inference_prompt_cost': 6e-07, 'upstream_inference_cost': 5e-06}}, id='lc_run--019f73a7-3bab-7983-ab0c-d4c14a76f6fb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 11, 'total_tokens': 17, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

# Free Model

In [20]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENROUTER_API_KEY"] = "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"

model = init_chat_model(
    "nvidia/nemotron-3-super-120b-a12b:free",
    model_provider="openrouter",
)

response = model.invoke("Hello Which model are you ?")

In [ ]:
response

AIMessage(content='I am **Nemotron 3 Super**, a large language model created by NVIDIA. How can I assist you today? 😊', additional_kwargs={'reasoning_content': "Okay, the user is asking which model I am. Let me recall that I'm Nemotron 3 Super, created by NVIDIA. I should state that clearly. The user might be checking if I'm a specific version or comparing models. They might want to know my capabilities or origin. I'll keep it straightforward and friendly. No need for extra details unless they ask follow-ups. Just confirm the model and offer help.\n", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "Okay, the user is asking which model I am. Let me recall that I'm Nemotron 3 Super, created by NVIDIA. I should state that clearly. The user might be checking if I'm a specific version or comparing models. They might want to know my capabilities or origin. I'll keep it straightforward and friendly. No need for extra details unless they ask follow-up

https://openrouter.ai/openrouter/free

In [21]:
import os
from langchain.chat_models import init_chat_model



model = init_chat_model(
    "openrouter/free",  # Automatic but for free
    model_provider="openrouter",
)

response = model.invoke("Hello Which model are you ?")

In [ ]:
model = init_chat_model(
    "claude-sonnet-4-6",
    # Kwargs passed to the model:
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,  # Default; increase for unreliable networks
)

In [22]:
response = model.invoke("Explain agentic AI in one sentence.")
print("text:             ", response.text)
print("content_blocks:   ", response.content_blocks)
print("id:               ", response.id)
print("tool_calls:       ", response.tool_calls)

text:              
Agentic AI refers to autonomous systems that perceive, learn, and plan to achieve specific goals by interacting with their environment, often adapting and making decisions with minimal human intervention.

content_blocks:    [{'type': 'reasoning', 'reasoning': '\nOkay, the user wants a one-sentence explanation of agentic AI. Let me start by recalling what agentic AI is. From what I remember, it\'s about AI systems that can act autonomously to achieve goals. But I need to make sure I have the right definition.\n\nAgentic AI probably refers to agents that can make decisions and take actions without human intervention. They might have some level of autonomy, maybe using machine learning to adapt or learn from their environment. I should mention goal-oriented behavior and the ability to interact with their environment. Also, they might use perception and planning to execute tasks.\n\nWait, the user specified one sentence. I need to be concise. Let me check if there\'s a

In [23]:
response

AIMessage(content='\nAgentic AI refers to autonomous systems that perceive, learn, and plan to achieve specific goals by interacting with their environment, often adapting and making decisions with minimal human intervention.\n', additional_kwargs={'reasoning_content': '\nOkay, the user wants a one-sentence explanation of agentic AI. Let me start by recalling what agentic AI is. From what I remember, it\'s about AI systems that can act autonomously to achieve goals. But I need to make sure I have the right definition.\n\nAgentic AI probably refers to agents that can make decisions and take actions without human intervention. They might have some level of autonomy, maybe using machine learning to adapt or learn from their environment. I should mention goal-oriented behavior and the ability to interact with their environment. Also, they might use perception and planning to execute tasks.\n\nWait, the user specified one sentence. I need to be concise. Let me check if there\'s a standard d

In [25]:
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage



In [26]:

ai_msg = AIMessage("I'd be happy to help you with that question!")
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Great! What's 2+2?"),
]
response = model.invoke(messages)
print(response.content)

The answer is **4**.


In [27]:
model =

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-openrouter': '0.2.6'}}, output_version=None, profile={'name': 'Free Models Router', 'release_date': '2026-02-01', 'last_updated': '2026-02-01', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 8000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x794de7717980>, openrouter_api_key=SecretStr('**********'), openrouter_api_base=None, app_url='https://docs.langchain.com', app_title='LangChain', model_name='openrouter/free', model_kwargs={}, session_id=None)

In [28]:
model.invoke("Hi, who are you ?")

AIMessage(content="Hello! I'm Nemotron-H, a large-scale language model independently developed by NVIDIA Group. I'm here to provide information, answer your questions, and assist you with various tasks. Whether you need knowledge, advice, or just friendly conversation, feel free to ask me anything! 😊 What can I help you with today?\n", additional_kwargs={}, response_metadata={'model_name': 'nvidia/nemotron-nano-12b-v2-vl:free', 'id': 'gen-1784431400-4TmPog1d6JwgSBjk17wN', 'created': 1784431400, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--019f7866-56e8-71e1-b1a3-e13ff70cce13-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 70, 'total_tokens': 90, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_deta

In [29]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]
# Exactly Same, no issues at all.
# messages = [
#     SystemMessage("You are a helpful assistant"),
#     HumanMessage("Can you help me?"),
#     ai_msg,
#     HumanMessage("Great! What's 2+2?"),
# ]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content="J'adore créer des applications." additional_kwargs={'reasoning_content': 'Need translation: "I love building applications." We\'ll produce French translation. Also consider nuance: "Je t\'aime construire des applications" or "Je suis passionné par la création d\'applications". But simplest "J\'aime construire des applications." Or "J\'aime créer des applications." Might be better: "J\'adore construire des applications." I\'ll produce "J\'adore créer des applications." Ensure correct.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Need translation: "I love building applications." We\'ll produce French translation. Also consider nuance: "Je t\'aime construire des applications" or "Je suis passionné par la création d\'applications". But simplest "J\'aime construire des applications." Or "J\'aime créer des applications." Might be better: "J\'adore construire des applications." I\'ll produce "J\'adore créer des applications." Ensure cor

In [30]:
messages.append(response)

# Streaming

In [22]:
openai_model = init_chat_model("openai:gpt-5-mini")

In [23]:
chunks = []
full_message = None

In [28]:
import time
for chunk in openai_model.stream("Write a long sentence about how AI Agents work"):
  chunks.append(chunk)
  print(repr(chunk.text),type(chunk))
  full_message = chunk if full_message is None else full_message + chunk
  time.sleep(1)

print()
print("Reconstructed full message:", full_message.content)


'' <class 'langchain_core.messages.ai.AIMessageChunk'>
'AI' <class 'langchain_core.messages.ai.AIMessageChunk'>
' agents' <class 'langchain_core.messages.ai.AIMessageChunk'>
' operate' <class 'langchain_core.messages.ai.AIMessageChunk'>
' by' <class 'langchain_core.messages.ai.AIMessageChunk'>
' perce' <class 'langchain_core.messages.ai.AIMessageChunk'>
'iving' <class 'langchain_core.messages.ai.AIMessageChunk'>
' an' <class 'langchain_core.messages.ai.AIMessageChunk'>
' environment' <class 'langchain_core.messages.ai.AIMessageChunk'>
' through' <class 'langchain_core.messages.ai.AIMessageChunk'>
' sensors' <class 'langchain_core.messages.ai.AIMessageChunk'>
' or' <class 'langchain_core.messages.ai.AIMessageChunk'>
' data' <class 'langchain_core.messages.ai.AIMessageChunk'>
' streams' <class 'langchain_core.messages.ai.AIMessageChunk'>
' and' <class 'langchain_core.messages.ai.AIMessageChunk'>
' encoding' <class 'langchain_core.messages.ai.AIMessageChunk'>
' that' <class 'langchain_cor

In [29]:
q1 = 'hi, how are you ?'
q2 = 'tell about AI in less than 100 words?'
q3 = 'Explain agents in 30 words ?'

In [30]:
responses = openai_model.batch([
    q1,
    q2,
    q3
])

In [31]:
for response in responses:
  print(response)

content='Hi — I’m an AI, so I don’t have feelings, but I’m ready to help. What can I do for you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 12, 'total_tokens': 114, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E3Ct9GCmggEXniXN7ihYQMroCKKaE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f7882-55e2-7e81-ac74-c3490a89054e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 12, 'output_tokens': 102, 'total_tokens': 114, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}}
content='Artificial intelligence 

In [32]:
for response in openai_model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

(0, AIMessage(content='Parrots are colorful for a mix of physical and evolutionary reasons. The bright hues come from two main feather mechanisms and the colors are useful for communication, survival, or both.\n\nHow the colors are made\n- Pigments: Parrots make unique pigments called psittacofulvins that produce red, orange and yellow shades. They also have melanins (browns/black) like other birds. Unlike many birds that rely on dietary carotenoids for reds/yellows, parrots synthesize psittacofulvins themselves.\n- Structural color: Blues and many greens aren’t from pigment but from microscopic feather structures that scatter light. Blue light is scattered more, producing blue. When a yellow pigment overlays structural blue, the result looks green.\n\nWhy those colors evolved\n- Mate choice/sexual selection: Bright, well-kept plumage signals health and genetic quality to potential mates. Colorful birds often have higher mating success.\n- Species, sex and individual recognition: Disti

# Tools & Models

In [34]:
openai_model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-openai': '1.3.5'}}, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7fe70e48e4e0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7fe70e48e600>, root_client=<openai.OpenAI object at 0x7fe70e48e5d0>, root_asyn

In [39]:
def get_weather(location:str) ->str:
  "Get the weather at a location"
  print("I was called from somewhere")
  return f"Sunny in '{location}"

In [44]:
def set_password(new_pass):
  "It is a tool to set password"
  return "Password changed"

In [45]:
model_with_tools = openai_model.bind_tools([get_weather,set_password])

In [51]:
response = model_with_tools.invoke('set password to admin123')

In [52]:
response.tool_calls

[{'name': 'set_password',
  'args': {'new_pass': 'admin123'},
  'id': 'call_beucsdRo5EHRWPgYrGnvhOXG',
  'type': 'tool_call'}]

# Structured Model Output

In [53]:
from pydantic import BaseModel,Field

In [55]:
class Email(BaseModel):
  subject: str = Field(description='The subject of the email')
  body: str = Field(description='The body of the email')

In [57]:
model_with_structure=openai_model.with_structured_output(Email)

In [58]:
response = model_with_structure.invoke('write a message to my manager for leave')

In [60]:
type(response)

__main__.Email

In [59]:
response

Email(subject='Leave Request: [Start Date] to [End Date]', body="Hi [Manager's Name],\n\nI would like to request leave from [Start Date] through [End Date] (inclusive) due to [brief reason — e.g., medical appointment / family matter / personal reasons / vacation]. I will complete/hand over the following before I go: [task 1 / task 2], and I have briefed [colleague name] to cover urgent issues. I will be reachable at [phone/email] for urgent matters if needed.\n\nPlease let me know if you need any additional information or documentation. Thank you for your consideration.\n\nBest regards,\n[Your Name]\n[Your Position]\n\nShort/informal version:\n\nHi [Manager's Name],\n\nI need to take leave from [Start Date] to [End Date] for [reason]. I’ve arranged coverage with [colleague] and will be available for urgent questions at [phone/email]. Please let me know if that’s okay.\n\nThanks,\n[Your Name]")

In [ ]:
#

# Messages

In [ ]:
from langchain.messages